# Parco Veicolare — Tasso di motorizzazione

Analisi del tasso di motorizzazione (autovetture per 1.000 abitanti) in Italia, per macro-area geografica e per i comuni capoluogo, con focus su **Bologna** e **Olbia** nel contesto dell'introduzione della Zona 30.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

In [ ]:
df_raw = pd.read_csv(
    '/content/drive/MyDrive/TesiMagistrale/FontiUsate/parco_veicolare.csv',
    sep=';',
    decimal=',',
    encoding='utf-8'
)

# Teniamo solo l'indicatore CPASCAR1000 e le colonne utili
df = df_raw[df_raw['DATA_TYPE'] == 'CPASCAR1000'][['Territorio', 'TIME_PERIOD', 'Osservazione']].copy()
df = df.rename(columns={'TIME_PERIOD': 'Anno', 'Osservazione': 'Tasso'})
df['Tasso'] = pd.to_numeric(df['Tasso'], errors='coerce')
df['Anno'] = df['Anno'].astype(int)

print(f'✅ Righe caricate: {len(df):,}')
print(f'Anni disponibili: {sorted(df["Anno"].unique())}')

In [ ]:
# Esplora i territori disponibili
print('Territori presenti nel dataset:')
for t in df['Territorio'].unique():
    print(' ', t)

In [ ]:
# Pivot: righe = Anno, colonne = Territorio
pivot = df.pivot_table(index='Anno', columns='Territorio', values='Tasso')
pivot.head()

In [ ]:
# ── Colori coerenti con Zone30.ipynb ────────────────────────────────────
C_BO = '#2166ac'   # blu  — Bologna / Nord-est
C_OL = '#d6604d'   # rosso-arancio — Olbia / Isole
C_IT = '#888888'   # grigio — Italia

# Anni di interesse (adatta in base ai dati disponibili)
anni_disponibili = sorted(df['Anno'].unique())

# ── Nomi macro-aree (verifica con la cella precedente se necessario) ────
MACRO_AREE = {
    'Italia':    {'color': C_IT,      'lw': 2.0, 'ls': '-'},
    'Nord-est':  {'color': C_BO,      'lw': 1.6, 'ls': '--'},
    'Nord-ovest':{'color': '#4393c3', 'lw': 1.6, 'ls': '--'},
    'Centro':    {'color': '#74c476', 'lw': 1.6, 'ls': '--'},
    'Sud':       {'color': '#fd8d3c', 'lw': 1.6, 'ls': '--'},
    'Isole':     {'color': C_OL,      'lw': 1.6, 'ls': '--'},
}

fig, ax = plt.subplots(figsize=(12, 6))

for nome, stile in MACRO_AREE.items():
    if nome in pivot.columns:
        ax.plot(
            pivot.index,
            pivot[nome],
            color=stile['color'],
            lw=stile['lw'],
            ls=stile['ls'],
            label=nome
        )

ax.set_title('Tasso di motorizzazione per macro-area geografica', fontsize=14)
ax.set_xlabel('Anno')
ax.set_ylabel('Autovetture per 1.000 abitanti')
ax.set_xticks(anni_disponibili)
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Bologna e Olbia vs macro-area di riferimento e Italia ───────────────
# Bologna → Nord-est; Olbia → Isole
# Adatta i nomi se il dataset usa varianti (es. 'Nord-Est' con maiuscola)

serie = {
    'Italia':   {'col': 'Italia',   'color': C_IT, 'lw': 1.5, 'ls': '-'},
    'Nord-est': {'col': 'Nord-est', 'color': C_BO, 'lw': 1.8, 'ls': '--'},
    'Isole':    {'col': 'Isole',    'color': C_OL, 'lw': 1.8, 'ls': '--'},
    'Bologna':  {'col': 'Bologna',  'color': C_BO, 'lw': 2.5, 'ls': '-'},
    'Olbia':    {'col': 'Olbia',    'color': C_OL, 'lw': 2.5, 'ls': '-'},
}

fig, ax = plt.subplots(figsize=(12, 6))

for label, s in serie.items():
    if s['col'] in pivot.columns:
        ax.plot(
            pivot.index,
            pivot[s['col']],
            color=s['color'],
            lw=s['lw'],
            ls=s['ls'],
            label=label
        )

ax.axvline(x=2023.5, color=C_BO, lw=1.2, ls=':', alpha=0.85, label='Zona 30 Bologna (gen 2024)')
ax.axvline(x=2021.5, color=C_OL, lw=1.2, ls=':', alpha=0.85, label='Zona 30 Olbia (giu 2021)')

ax.set_title('Tasso di motorizzazione: Bologna e Olbia vs macro-area e Italia', fontsize=14)
ax.set_xlabel('Anno')
ax.set_ylabel('Autovetture per 1.000 abitanti')
ax.set_xticks(anni_disponibili)
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Tabella riepilogativa: valori per le aree chiave
cols_chiave = [c for c in ['Italia', 'Nord-est', 'Isole', 'Bologna', 'Olbia'] if c in pivot.columns]
pivot[cols_chiave].dropna(how='all')

---
## Ringiovanimento del parco auto

Quota percentuale di autovetture con **meno di 8 anni** (fasce LES1YEARCAR + 1TO3YEARCAR + 4TO7YEARCAR) sul totale del parco circolante.

Una quota crescente di auto recenti è un potenziale **fattore di confondimento** nell'analisi della Zona 30: le auto più nuove sono mediamente più sicure (frenata, sistemi di assistenza alla guida, struttura delle carrozzerie), quindi un ringiovanimento del parco potrebbe ridurre la gravità degli incidenti indipendentemente dalla policy.

In [ ]:
# ── Carica le 3 fasce "giovani" (disponibili per tutti gli anni 2015–2024) ──
GIOVANI_TYPES = ['LES1YEARCAR', '1TO3YEARCAR', '4TO7YEARCAR']

df_age = df_raw[df_raw['DATA_TYPE'].isin(GIOVANI_TYPES)][
    ['Territorio', 'DATA_TYPE', 'TIME_PERIOD', 'Osservazione']
].copy()
df_age = df_age.rename(columns={'TIME_PERIOD': 'Anno', 'Osservazione': 'Valore'})
df_age['Valore'] = pd.to_numeric(df_age['Valore'], errors='coerce')
df_age['Anno'] = df_age['Anno'].astype(int)

# Somma le 3 fasce per area/anno → quota auto ≤7 anni
quota_giovani = (
    df_age
    .groupby(['Territorio', 'Anno'])['Valore']
    .sum()
    .reset_index()
    .rename(columns={'Valore': 'QuotaGiovani'})
)
piv_g = quota_giovani.pivot_table(index='Anno', columns='Territorio', values='QuotaGiovani')
anni_g = sorted(piv_g.index.tolist())

print(f'✅ Anni disponibili: {anni_g}')

In [ ]:
# ── Grafico: quota auto ≤7 anni ─────────────────────────────────────────
SERIE_G = [
    ('Italia',   C_IT, 1.5, '-'),
    ('Nord-est', C_BO, 1.8, '--'),
    ('Isole',    C_OL, 1.8, '--'),
    ('Bologna',  C_BO, 2.5, '-'),
    ('Olbia',    C_OL, 2.5, '-'),
]

fig, ax = plt.subplots(figsize=(12, 6))

for label, color, lw, ls in SERIE_G:
    if label in piv_g.columns:
        ax.plot(anni_g, piv_g.loc[anni_g, label], color=color, lw=lw, ls=ls, label=label)

ax.axvline(x=2023.5, color=C_BO, lw=1.2, ls=':', alpha=0.85, label='Zona 30 Bologna (gen 2024)')
ax.axvline(x=2021.5, color=C_OL, lw=1.2, ls=':', alpha=0.85, label='Zona 30 Olbia (giu 2021)')

ax.set_title('Quota di auto con meno di 8 anni sul totale del parco (2015\u20132024)', fontsize=14)
ax.set_xlabel('Anno')
ax.set_ylabel('% autovetture < 8 anni')
ax.set_xticks(anni_g)
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f%%'))
ax.legend(framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()